# HITL Review Interface

Interactive human-in-the-loop review for low-confidence OCR words. The widget
walks through uncertain words one at a time, showing the word in context on the
document image and updating a live transcription as corrections are made.

In [ ]:
%matplotlib widget
import json
from pathlib import Path

import cv2
import numpy as np
import yaml
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display

from scribe.preprocessing import analyze_image, preprocess
from scribe.ocr import run_tesseract
from scribe.visualization import render_ocr_overlay

project_root = Path("..").resolve()
config = yaml.safe_load(open(project_root / "configs" / "pipeline.yaml"))
manifest = yaml.safe_load(open(project_root / "data" / "images.yaml"))["images"]
images_dir = project_root / "data" / "images"

# Load and process grocery_contract
meta = manifest["grocery_contract"]
img_bgr = cv2.imread(str(images_dir / meta["filename"]))
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
recipe = analyze_image(gray, config)
preprocessed = preprocess(gray, recipe)
ocr_result = run_tesseract(preprocessed)
words = ocr_result["words"]

# Also keep the raw gray for the toggle
img_original = gray
img_preprocessed = preprocessed

print(f"Loaded grocery_contract: {len(words)} words, avg confidence {ocr_result['avg_confidence']:.1f}%")
print(f"Words below 60% confidence: {sum(1 for w in words if w['conf'] < 60)}")

In [ ]:
# ── State ──
CONF_THRESHOLD = 60  # words below this go into the review queue
review_queue = []    # indices into `words` list
current_idx = [0]    # mutable container for closure access

def rebuild_queue(sort_mode):
    """Rebuild the review queue based on sort mode."""
    candidates = [(i, w) for i, w in enumerate(words) if w["conf"] < CONF_THRESHOLD]
    if sort_mode == "Lowest confidence first":
        candidates.sort(key=lambda x: x[1]["conf"])
    else:
        # Reading order: top-to-bottom, left-to-right
        candidates.sort(key=lambda x: (x[1]["bbox"]["y"], x[1]["bbox"]["x"]))
    review_queue.clear()
    review_queue.extend([i for i, _ in candidates])
    current_idx[0] = 0

# ── Figure: crop on top, document + transcription below ──
fig = plt.figure(figsize=(18, 14))
ax_crop = fig.add_axes([0.05, 0.75, 0.9, 0.22])   # top: zoomed crop
ax_img  = fig.add_axes([0.02, 0.02, 0.47, 0.70])   # bottom-left: full document
ax_text = fig.add_axes([0.51, 0.02, 0.47, 0.70])   # bottom-right: transcription

# Crop padding: multiplier on word dimensions
CROP_PAD_X = 1.5  # horizontal padding as multiple of word width
CROP_PAD_Y = 1.5  # vertical padding as multiple of word height
CROP_MIN_X = 50   # minimum horizontal padding in pixels
CROP_MIN_Y = 20   # minimum vertical padding in pixels

def conf_color(c):
    if c >= 80: return (0, 0.55, 0)
    elif c >= 50: return (0.8, 0.5, 0)
    else: return (0.8, 0, 0)

def draw_crop(img):
    """Draw a zoomed crop around the current review word."""
    ax_crop.clear()
    if not review_queue or current_idx[0] >= len(review_queue):
        ax_crop.set_facecolor("#f0f0f0")
        ax_crop.text(0.5, 0.5, "No word selected", ha="center", va="center",
                     fontsize=14, color="gray", transform=ax_crop.transAxes)
        ax_crop.axis("off")
        return

    wi = review_queue[current_idx[0]]
    w = words[wi]
    b = w["bbox"]
    h_img, w_img = img.shape[:2]

    pad_x = max(int(b["w"] * CROP_PAD_X), CROP_MIN_X)
    pad_y = max(int(b["h"] * CROP_PAD_Y), CROP_MIN_Y)
    x0 = max(0, b["x"] - pad_x)
    y0 = max(0, b["y"] - pad_y)
    x1 = min(w_img, b["x"] + b["w"] + pad_x)
    y1 = min(h_img, b["y"] + b["h"] + pad_y)

    crop = img[int(y0):int(y1), int(x0):int(x1)]
    ax_crop.imshow(crop, cmap="gray")

    # Draw the bbox on the crop (shifted to crop coordinates)
    rect = patches.Rectangle(
        (b["x"] - x0, b["y"] - y0), b["w"], b["h"],
        linewidth=3, edgecolor=(0, 0.4, 1), facecolor=(0, 0.4, 1, 0.1)
    )
    ax_crop.add_patch(rect)
    ax_crop.set_title(f'Current word: "{w["text"]}"  (confidence: {w["conf"]}%)', fontsize=13)
    ax_crop.axis("off")

def draw_image(img_source):
    """Draw the full document image with only the current word highlighted."""
    ax_img.clear()
    img = img_preprocessed if img_source == "Preprocessed" else img_original
    ax_img.imshow(img, cmap="gray")

    # Only draw one box — the current review word
    if review_queue and current_idx[0] < len(review_queue):
        wi = review_queue[current_idx[0]]
        b = words[wi]["bbox"]
        rect = patches.Rectangle(
            (b["x"], b["y"]), b["w"], b["h"],
            linewidth=2, edgecolor=(0, 0.4, 1), facecolor="none"
        )
        ax_img.add_patch(rect)

    ax_img.set_title(f"Document ({img_source.lower()})", fontsize=12)
    ax_img.axis("off")

def draw_transcription():
    """Draw the text transcription colored by confidence."""
    ax_text.clear()
    ax_text.set_xlim(0, img_preprocessed.shape[1])
    ax_text.set_ylim(img_preprocessed.shape[0], 0)

    for w in words:
        b = w["bbox"]
        color = conf_color(w["conf"])
        fontsize = max(5, min(10, b["h"] * 0.4))
        ax_text.text(b["x"], b["y"] + b["h"] * 0.8, w["text"],
                     fontsize=fontsize, color=color, family="monospace",
                     clip_on=True)

    ax_text.set_facecolor("white")
    ax_text.set_title("Transcription (updates with corrections)", fontsize=12)
    ax_text.axis("off")

def refresh_display():
    img_source = w_img_source.value
    img = img_preprocessed if img_source == "Preprocessed" else img_original
    draw_crop(img)
    draw_image(img_source)
    draw_transcription()

    # Update info panel for the NEW current word
    if review_queue and current_idx[0] < len(review_queue):
        wi = review_queue[current_idx[0]]
        w = words[wi]
        w_conf_label.value = f"Confidence: {w['conf']}%"
        w_progress.value = f"Word {current_idx[0] + 1} of {len(review_queue)} remaining"
        # Set the text box LAST so it doesn't trigger on_submit
        w_word_input.value = w["text"]
    else:
        w_word_input.value = ""
        w_conf_label.value = "Done!"
        w_progress.value = f"All {len(review_queue)} words reviewed"

    fig.canvas.draw_idle()

# ── Controls ──
w_img_source = widgets.RadioButtons(
    options=["Preprocessed", "Original"], value="Preprocessed",
    description="Image:", layout=widgets.Layout(width="200px")
)

w_sort = widgets.Dropdown(
    options=["Lowest confidence first", "Reading order"],
    value="Lowest confidence first",
    description="Sort by:", layout=widgets.Layout(width="300px")
)

w_word_input = widgets.Text(description="Word:", layout=widgets.Layout(width="300px"))
w_conf_label = widgets.Label(value="")
w_progress = widgets.Label(value="")

w_accept = widgets.Button(description="Accept ✓", button_style="success",
                           layout=widgets.Layout(width="120px"))
w_correct = widgets.Button(description="Correct →", button_style="warning",
                            layout=widgets.Layout(width="120px"))
w_skip = widgets.Button(description="Skip ↓", button_style="info",
                         layout=widgets.Layout(width="120px"))

def apply_and_advance():
    """Apply current text box value as correction, then advance."""
    if review_queue and current_idx[0] < len(review_queue):
        wi = review_queue[current_idx[0]]
        new_text = w_word_input.value.strip()
        if new_text:
            words[wi]["text"] = new_text
        words[wi]["conf"] = 100  # mark as human-reviewed
    current_idx[0] = min(current_idx[0] + 1, len(review_queue))
    refresh_display()

def on_accept(b):
    """Accept the current word as-is and move to next."""
    apply_and_advance()

def on_correct(b):
    """Apply the correction from the text box and move to next."""
    apply_and_advance()

def on_submit(sender):
    """Enter key in the text box applies correction and advances."""
    apply_and_advance()

def on_skip(b):
    current_idx[0] = min(current_idx[0] + 1, len(review_queue))
    refresh_display()

def on_sort_change(change):
    rebuild_queue(change["new"])
    refresh_display()

def on_img_source_change(change):
    refresh_display()

w_word_input.on_submit(on_submit)
w_accept.on_click(on_accept)
w_correct.on_click(on_correct)
w_skip.on_click(on_skip)
w_sort.observe(on_sort_change, names="value")
w_img_source.observe(on_img_source_change, names="value")

# ── Layout ──
controls_top = widgets.HBox([w_img_source, w_sort])
controls_bottom = widgets.HBox([
    w_word_input, w_conf_label,
    w_accept, w_correct, w_skip,
    w_progress,
])

# ── Initialize ──
rebuild_queue(w_sort.value)
display(widgets.VBox([controls_top, controls_bottom]))
refresh_display()